# KV cache 代码导读

从 [MiniMind 的 model_minimind.py](https://github.com/jingyaogong/minimind/blob/233d00fa068ec2b8c0d4f07b703934d6391a5a73/model/model_minimind.py) 入手，再对照 Hugging Face 的 Cache API 和 vLLM 的分页存储。顺序执行各单元格，观察每次 forward 的输入长度、每层 KV 的形状和 logits 对照结果。

**运行环境**。选择已安装 PyTorch 和 Hugging Face Transformers 的 Python 内核。可在 Notebook 中用 `%pip install torch transformers` 安装。示例用 CPU、随机权重和整数 token ID，无需 tokenizer、模型权重或 GPU。 本笔记用 Python 3.12、PyTorch 2.15.0 和 Transformers 5.22.2 验证。第一个代码单元会下载并导入上面固定提交的 MiniMind 源码。已有该版本的源码时，可通过环境变量 `MINIMIND_SOURCE` 指向本地 `model_minimind.py`，随后可以离线运行。

## 先跟一遍 MiniMind 的调用栈

以下行号基于上面的固定版本。遇到不同版本时，按函数名找入口。

| 顺序 | 源码入口 | 读代码时盯住什么 |
| --- | --- | --- |
| 1 | [`MiniMindForCausalLM.generate`](https://github.com/jingyaogong/minimind/blob/233d00fa068ec2b8c0d4f07b703934d6391a5a73/model/model_minimind.py#L262) | 从缓存读 `past_len`，只送入 `input_ids[:, past_len:]`。从本轮最后一个位置的 logits 选新 token，再把本轮的缓存交给下一轮。 |
| 2 | [`MiniMindForCausalLM.forward`](https://github.com/jingyaogong/minimind/blob/233d00fa068ec2b8c0d4f07b703934d6391a5a73/model/model_minimind.py#L250) | 调用模型主体，拿 hidden state 做 `lm_head`，通过 `outputs.past_key_values` 返回各层缓存。 |
| 3 | [`MiniMindModel.forward`](https://github.com/jingyaogong/minimind/blob/233d00fa068ec2b8c0d4f07b703934d6391a5a73/model/model_minimind.py#L214) | 用第一层缓存的序列长度确定 `start_pos`，切出 `[start_pos, start_pos + T_new)` 的 RoPE cos/sin，再逐层传入对应的旧 KV。 |
| 4 | [`MiniMindBlock.forward`](https://github.com/jingyaogong/minimind/blob/233d00fa068ec2b8c0d4f07b703934d6391a5a73/model/model_minimind.py#L191) | 输入 RMSNorm、Attention、输出侧 RMSNorm 和 MLP 都只处理这轮送进来的位置。 |
| 5 | [`Attention.forward`](https://github.com/jingyaogong/minimind/blob/233d00fa068ec2b8c0d4f07b703934d6391a5a73/model/model_minimind.py#L111) | 投影新 Q/K/V → 对 Q/K 归一化并应用 RoPE → 沿时间轴拼接旧 K/V → 保存拼接结果 → 为 Attention 扩展 KV 头。 |

`past_key_values[layer_idx]` 是一层的 `(K, V)`。MiniMind 存储形状为 **`[B, T_total, H_kv, D_head]`**，所以拼接用 `dim=1`。K 已经应用 RoPE，V 不旋转。缓存保留的是 GQA 的少量 KV 头，`repeat_kv` 在保存缓存之后才把它们对齐到 Q 头数。模型里的 RMSNorm 和 FFN 都逐位置运算，旧位置可以跳过，因为后续 Attention 只需要它们已经算好的 K/V。

In [ ]:
import importlib.util
import os
from pathlib import Path
from tempfile import mkdtemp
from urllib.request import urlretrieve

import torch
import transformers

revision = "233d00fa068ec2b8c0d4f07b703934d6391a5a73"
local_source = os.environ.get("MINIMIND_SOURCE")
if local_source:
    source_path = Path(local_source).expanduser()
else:
    source_path = Path(mkdtemp(prefix="kv-cache-minimind-")) / "model_minimind.py"
    source_url = f"https://raw.githubusercontent.com/jingyaogong/minimind/{revision}/model/model_minimind.py"
    urlretrieve(source_url, source_path)

spec = importlib.util.spec_from_file_location("model_minimind", source_path)
minimind = importlib.util.module_from_spec(spec)
spec.loader.exec_module(minimind)

torch.manual_seed(7)
model = minimind.MiniMindForCausalLM(minimind.MiniMindConfig(
    hidden_size=32,
    num_hidden_layers=2,
    num_attention_heads=4,
    num_key_value_heads=2,
    intermediate_size=64,
    max_position_embeddings=64,
    vocab_size=64,
)).eval()
prompt = torch.tensor([[5, 12, 23, 8]])
print(f"PyTorch {torch.__version__}, Transformers {transformers.__version__}")
print(f"MiniMind 源码位置 {source_path}")

## 手动跑一次 prefill，再接一次 decode

输入 4 个 prompt token 后，最后一个位置的 logits 用来选第一个生成 token。此时缓存长度仍是 4。**选出了一个 token 和处理完一个 token 是两件事**。把这个 token 喂回模型之后，缓存长度才增长到 5，同时得到下一个 token 的 logits。

下面用完整重算作参照。等价性用 `assert_close` 检查，允许不同算子路径带来的浮点舍入差异。

In [ ]:
with torch.inference_mode():
    prefill = model(prompt, use_cache=True)
    next_token = prefill.logits[:, -1].argmax(dim=-1, keepdim=True)
    decode = model(next_token, past_key_values=prefill.past_key_values, use_cache=True)
    full_ids = torch.cat((prompt, next_token), dim=1)
    reference = model(full_ids, use_cache=False)

for i, ((old_k, old_v), (new_k, new_v)) in enumerate(
    zip(prefill.past_key_values, decode.past_key_values)
):
    print(f"第 {i} 层 K/V {tuple(old_k.shape)} -> {tuple(new_k.shape)}")
    assert new_k.shape == new_v.shape == (1, 5, 2, 8)
    torch.testing.assert_close(new_k[:, :4], old_k, rtol=0, atol=0)
    torch.testing.assert_close(new_v[:, :4], old_v, rtol=0, atol=0)

torch.testing.assert_close(decode.logits, reference.logits[:, -1:], rtol=1e-4, atol=1e-5)
print("单 token decode 与完整重算的 logits 一致")

## 看 generate 实际往模型送了什么

在第一层 Attention 和 FFN 上挂观察函数，只记录它们每次收到多少位置。输入长度为 4、固定生成 3 个 token 时，有缓存应看到 `[4, 1, 1]`，关闭缓存应看到 `[4, 5, 6]`。本例关闭采样、过滤和 EOS 停止，便于对比。

In [ ]:
def trace_generate(use_cache):
    first_block = model.model.layers[0]
    lengths = {"attention": [], "ffn": []}
    handles = [
        first_block.self_attn.register_forward_pre_hook(
            lambda module, args: lengths["attention"].append(args[0].shape[1])
        ),
        first_block.mlp.register_forward_pre_hook(
            lambda module, args: lengths["ffn"].append(args[0].shape[1])
        ),
    ]
    try:
        result = model.generate(
            inputs=prompt,
            use_cache=use_cache,
            max_new_tokens=3,
            do_sample=False,
            top_k=0,
            top_p=1.0,
            eos_token_id=None,
            return_kv=True,
        )
    finally:
        for handle in handles:
            handle.remove()
    return result, lengths

cached, cached_lengths = trace_generate(True)
uncached, uncached_lengths = trace_generate(False)
print("有缓存", cached_lengths)
print("无缓存", uncached_lengths)
assert cached_lengths == {"attention": [4, 1, 1], "ffn": [4, 1, 1]}
assert uncached_lengths == {"attention": [4, 5, 6], "ffn": [4, 5, 6]}
torch.testing.assert_close(cached["generated_ids"], uncached["generated_ids"])
assert cached["past_kv"][0][0].shape[1] == cached["generated_ids"].shape[1] - 1

## 接两个新 token 时，mask 应该对齐哪一边？

缓存了 4 个位置，再送入 2 个新位置时，Attention 分数形状是 `[B, H_q, 2, 6]`。两个 Q 的绝对位置分别是 4 和 5。位置 4 可以看 0 到 4，位置 5 可以看 0 到 5。MiniMind 的手写分支把三角 mask 加在最右侧的两列，左边 4 个历史位置对这两个 Q 都可见。

PyTorch SDPA 的 `is_causal=True` 对非方阵默认按左上角对齐。接已有缓存时，可以用显式的绝对位置 mask（下面的 `allowed`），或使用适合该布局的右下角因果 bias。本例没有 padding。若传入 MiniMind 的 `attention_mask`，它的长度需要覆盖历史位置和新位置，并把 padding key 单独屏蔽。 [PyTorch SDPA 文档](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.scaled_dot_product_attention.html)

In [ ]:
suffix = torch.tensor([[9, 15]])
past_len = prompt.shape[1]
query_positions = past_len + torch.arange(suffix.shape[1])
key_positions = torch.arange(past_len + suffix.shape[1])
allowed = key_positions[None, :] <= query_positions[:, None]
print(allowed.int())

with torch.inference_mode():
    chunk = model(suffix, past_key_values=prefill.past_key_values, use_cache=True)
    full_chunk = model(torch.cat((prompt, suffix), dim=1), use_cache=False)

torch.testing.assert_close(chunk.logits, full_chunk.logits[:, -2:], rtol=1e-4, atol=1e-5)

## 改一个 token，检查是哪一层的 KV 变了

把 prompt 的第二个 token 替换掉，后两个位置和 token ID 保持不变。对照运行两次独立 prefill，逐层检查它们的 KV。第一层的后两个 K/V 都相同，第二层通常会变。第一个位置看不到改动，各层 KV 都相同。

In [ ]:
edited_prompt = prompt.clone()
edited_prompt[0, 1] = 31
with torch.inference_mode():
    edited = model(edited_prompt, use_cache=True)

for i, (before, after) in enumerate(zip(prefill.past_key_values, edited.past_key_values)):
    for name, old, new in zip(("K", "V"), before, after):
        torch.testing.assert_close(old[:, :1], new[:, :1], rtol=0, atol=0)
        tail_diff = (old[:, 2:] - new[:, 2:]).abs().max().item()
        print(f"第 {i} 层后缀 {name} 的最大差值 {tail_diff:.6f}")
        if i == 0:
            assert tail_diff == 0
        else:
            assert tail_diff > 0

## Hugging Face Transformers 的 DynamicCache 和 StaticCache

MiniMind 在这份源码里用每层一个 tuple 的列表。下面换到 Hugging Face 原生 Llama 实现，它把 KV 更新交给每层的 `Cache.update`，常见的 KV 排列为 `[B, H_kv, T, D_head]`。可以从 [`LlamaAttention.forward`](https://github.com/huggingface/transformers/blob/main/src/transformers/models/llama/modeling_llama.py) 跳到 [`cache_utils.py`](https://github.com/huggingface/transformers/blob/main/src/transformers/cache_utils.py)，找 `DynamicLayer.update` 和 `StaticLayer.update`。

| 缓存 | 如何写入 | 适用场景 |
| --- | --- | --- |
| `DynamicCache` | 普通全注意力层按需增长，拼接新 K/V | 通用生成的常见默认值，长度未知时容易使用 |
| `StaticCache` | 按最大长度分配固定形状的缓冲区，把新 KV 写进对应槽位 | 固定形状更适合编译或捕获图，代价是预留空间，过大的容量也可能带来额外 Attention 工作 |

日常可以让 `generate()` 管理缓存，需要固定缓存时，在支持的模型上设置 `cache_implementation="static"`。手动调用 `forward` 时，像下面一样只传未处理的 token，并为每条独立序列创建自己的缓存。示例用 Transformers 5.x 的接口，容量设为 prompt 长度加上续接长度。静态缓存张量的容量和已经写入的序列长度是两个值。 [Hugging Face 缓存策略](https://huggingface.co/docs/transformers/main/en/kv_cache)

In [ ]:
from transformers import DynamicCache, LlamaConfig, LlamaForCausalLM, StaticCache

hf_model = LlamaForCausalLM(LlamaConfig(
    hidden_size=32,
    intermediate_size=64,
    num_hidden_layers=2,
    num_attention_heads=4,
    num_key_value_heads=2,
    vocab_size=64,
    max_position_embeddings=64,
)).eval()

with torch.inference_mode():
    expected = hf_model(torch.cat((prompt, suffix), dim=1), use_cache=False).logits[:, -2:]
    caches = [
        DynamicCache(config=hf_model.config),
        StaticCache(config=hf_model.config, max_cache_len=8),
    ]
    for cache in caches:
        hf_model(prompt, past_key_values=cache, use_cache=True)
        current = hf_model(suffix, past_key_values=cache, use_cache=True)
        print(type(cache).__name__, "有效长度", int(cache.get_seq_length()),
              "第 0 层 K 形状", tuple(cache.layers[0].keys.shape))
        assert int(cache.get_seq_length()) == 6
        torch.testing.assert_close(current.logits, expected, rtol=1e-4, atol=1e-5)

## vLLM 如何在服务里管理 KV？

MiniMind 每步把历史 KV 和新 KV 拼起来。单条序列容易理解，但在并发服务里，每条请求的长度、结束时机和可复用的前缀都不同。vLLM 用 KV 块池和块表，把逻辑序列切成块，再映射到显存中的物理块。新 token 写到分配好的槽位，Attention 内核通过块表读对应历史 KV，无需为每条请求每步拼出整段连续张量。

举例，假设每块 4 个位置，某请求的块表是 `[5, 12]`。逻辑位置 6 属于第 1 个逻辑块，块内偏移 2，对应物理块 12 的第 2 号槽位。真实缓存的张量布局随 Attention 后端而异。

| 源码入口 | 建议跟读顺序 |
| --- | --- |
| [`block_pool.py`](https://github.com/vllm-project/vllm/blob/main/vllm/v1/core/block_pool.py) | 看物理块怎么分配、释放以及被前缀缓存命中。 |
| [`block_table.py`](https://github.com/vllm-project/vllm/blob/main/vllm/v1/worker/block_table.py) | 看逻辑位置如何变成 slot mapping，再对照 Attention 如何读历史块。 |
| [vLLM 前缀缓存设计](https://docs.vllm.ai/en/latest/design/prefix_caching/) | 看跨请求如何通过父块哈希共享完整块。分页解决存放和寻址，前缀匹配决定哪些块能直接共享。 |

普通全注意力模型的 KV 存储量可以先用 `2 × B × 层数 × T × KV 头数 × 每头维度 × 每元素字节数` 估算。前面的 2 是 K 和 V。实际分配还要考虑分页取整、数据类型和后端布局。缓存后单步 decode 只有少量新 Q，但全注意力仍需读取并访问历史 KV。GQA、较低精度的 KV 和高效 Attention 内核分别从头数、存储和访问方式降低成本。 [PagedAttention 论文](https://arxiv.org/abs/2309.06180)

## 自己过一遍

1. 在 `generate` 的 forward 调用前设断点，跟踪 `past_len`、新输入长度和本轮选出的 token。解释为什么最后一次返回的 KV 比生成序列短一个位置。
2. 在 `Attention.forward` 中比较 RoPE 前后的 K，再比较 `past_kv` 和传给 Attention 的 K。找出序列轴和 KV 头轴各在哪里变化。
3. 把两 token 的续接实验改成三 token。先画出预期的可见矩阵，再对照打印结果和 logits 断言。
4. 把“改一个 token”的实验移到第三个位置，预测哪几行的各层 KV 仍然精确相同。再读 [CacheBlend](CacheBlend.md)，看跨请求如何近似复用前文已经变化的后缀。